# Experiment 10: Hyperparameter Tuning and Optimization

**Objective:** Understand and apply hyperparameter tuning and optimization techniques to improve machine learning model performance using Python and Scikit-learn.

**Dataset:** Breast Cancer Wisconsin Diagnostic dataset bundled with Scikit-learn. This is a binary classification task. The notebook compares a baseline Support Vector Classifier (SVC) with models tuned using `GridSearchCV` and `RandomizedSearchCV`.

**Tools:** Python, NumPy, pandas, Matplotlib, Scikit-learn.

> Run the cells from top to bottom in Google Colab. The dataset is loaded from Scikit-learn and does not require a separate download.

## 1. Import libraries and load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from scipy.stats import loguniform

RANDOM_STATE = 42

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

print("Feature matrix:", X.shape)
print("Class names:", list(cancer.target_names))
print("Class counts:")
print(y.value_counts().rename(index=dict(enumerate(cancer.target_names))))
X.head()

## 2. Create training and test sets
The test set is held out until final evaluation. Stratification keeps the class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 3. Fit a baseline model
Scaling and classification are combined in a pipeline so the scaler is fitted only on each training fold during cross-validation.

In [ ]:
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

def metric_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }

results = [metric_row("Baseline SVC", y_test, baseline_pred)]
pd.DataFrame(results).set_index("Model").round(4)

## 4. Tune with GridSearchCV
Grid search evaluates every specified parameter combination using stratified 5-fold cross-validation. The scoring metric is F1, which balances precision and recall.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid_params = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.001, 0.01, 0.1],
    "svc__kernel": ["rbf", "linear"]
}

grid_search = GridSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("svc", SVC())]),
    param_grid=grid_params,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)
grid_search.fit(X_train, y_train)
print("Best CV F1:", round(grid_search.best_score_, 4))
print("Best parameters:", grid_search.best_params_)

## 5. Tune with RandomizedSearchCV
Randomized search samples from parameter distributions and can explore a larger space with a set compute budget. Here, `C` and `gamma` are sampled on a logarithmic scale.

In [ ]:
random_params = {
    "svc__C": loguniform(1e-2, 1e3),
    "svc__gamma": loguniform(1e-5, 1e0),
    "svc__kernel": ["rbf", "linear"]
}

random_search = RandomizedSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("svc", SVC())]),
    param_distributions=random_params,
    n_iter=30,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    return_train_score=True
)
random_search.fit(X_train, y_train)
print("Best CV F1:", round(random_search.best_score_, 4))
print("Best parameters:", random_search.best_params_)

## 6. Compare final test-set performance
Each selected estimator is refitted on the training data by the search object. We evaluate each once on the untouched test set.

In [ ]:
grid_pred = grid_search.predict(X_test)
random_pred = random_search.predict(X_test)

results = [
    metric_row("Baseline SVC", y_test, baseline_pred),
    metric_row("GridSearchCV SVC", y_test, grid_pred),
    metric_row("RandomizedSearchCV SVC", y_test, random_pred),
]
comparison = pd.DataFrame(results).set_index("Model")
comparison.round(4)

In [ ]:
ax = comparison[["Accuracy", "Precision", "Recall", "F1"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0.85, 1.0), rot=0
)
ax.set_ylabel("Test-set score")
ax.set_title("Baseline and tuned SVC performance")
ax.grid(axis="y", alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Inspect the best model
The grid search estimator is used here. If randomized search has a higher cross-validation F1, change `best_model` to `random_search.best_estimator_`.

In [ ]:
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)

print("Classification report (GridSearchCV model):")
print(classification_report(y_test, best_pred, target_names=cancer.target_names))
print("Confusion matrix:")
print(confusion_matrix(y_test, best_pred))

## 8. Examine search results
The table below shows the top grid-search configurations by mean cross-validation F1 score.

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "param_svc__C", "param_svc__gamma", "param_svc__kernel"]
cv_results[cols].sort_values("rank_test_score").head(10).reset_index(drop=True).round(4)

## 9. Conclusion and observations

Record your run's values from the comparison table and answer:

1. Which model achieved the highest test F1 score?
2. What hyperparameter values were selected by each search method?
3. Did tuning improve performance compared with the baseline?
4. How do GridSearchCV and RandomizedSearchCV differ in search effort and coverage?
5. Why should the test set not be used to choose hyperparameters?

**Expected conclusion:** Cross-validation helps select hyperparameters using only the training data. The final test set provides an independent estimate of generalization performance. The best score can vary with software versions and the randomized sample; report the values generated by your Colab run.

## Experiment record
**Name:** ____________________  
**Roll number:** ____________________  
**Date:** ____________________  
**Best method:** ____________________  
**Best test F1:** ____________________